# HaloScope

Use a weighted SVD projection to assign pseudo-labels, then fit a classifier to intermediate token activations. The experiment adapts the method to token-level code defect detection. [Paper](https://arxiv.org/abs/2409.17504)

Train an independent probe on each benchmark. Line and sample scores use the maximum annotated token score. F1 thresholds come from validation.

In [1]:
from pathlib import Path
import sys

if Path("traces.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import pandas as pd
import torch
from IPython.display import display

from experiments.data import DATASETS, PILOT_LIMITS
from experiments.evaluation import TraceStore, evaluate
from experiments.methods import HaloScope

pilot = False
limits = PILOT_LIMITS if pilot else None
torch.set_num_threads(min(4, torch.get_num_threads()))
store = TraceStore(quantize=True)

In [3]:
results = []
for benchmark in DATASETS:
    splits = store.load_splits(benchmark, limits)
    results.extend(
        {"benchmark": benchmark, **row}
        for row in evaluate(HaloScope(), splits)
    )

metrics = pd.DataFrame(results).set_index(["benchmark", "level"])
display(metrics.round(4))

n  positives  positive_rate  pr_auc  \
benchmark      level                                            
mbpp           token   3373        660         0.1957  0.1414   
               line     509         54         0.1061  0.0718   
               sample    74         31         0.4189  0.3441   
humaneval      token   3714        665         0.1791  0.1634   
               line     484         55         0.1136  0.0862   
               sample    46         19         0.4130  0.3960   
humaneval_java token   1157        205         0.1772  0.1449   
               line     155         15         0.0968  0.0662   
               sample    36         11         0.3056  0.2131   
defects4j      token   3901       2220         0.5691  0.4912   
               line     448        148         0.3304  0.2083   
               sample    86         35         0.4070  0.4375   

                       average_precision  roc_auc  precision  recall      f1  \
benchmark      level                                                           
mbpp           token              0.1416   0.3505     0.1957  1.0000  0.3274   
               line               0.0732   0.2961     0.1078  0.9259  0.1931   
               sample             0.3573   0.4044     0.4189  1.0000  0.5905   
humaneval      token              0.1637   0.4950     0.1792  1.0000  0.3039   
               line               0.0880   0.3944     0.1139  0.6545  0.1941   
               sample             0.4269   0.5068     0.4250  0.8947  0.5763   
humaneval_java token              0.1458   0.4390     0.1791  0.9951  0.3036   
               line               0.0708   0.3045     0.1034  1.0000  0.1875   
               sample             0.2334   0.2964     0.2812  0.8182  0.4186   
defects4j      token              0.4914   0.4289     0.5689  0.9991  0.7250   
               line               0.2097   0.1784     0.3311  1.0000  0.4975   
               sample             0.4497   0.5168     0.4177  0.9429  0.5789   

                       threshold  
benchmark      level              
mbpp           token      0.0000  
               line       0.0000  
               sample     0.9998  
humaneval      token      0.0000  
               line       0.0073  
               sample     1.0000  
humaneval_java token      0.0000  
               line       0.0041  
               sample     0.9998  
defects4j      token      0.0000  
               line       0.0000  
               sample     0.9999